---
title: Iskay Quantum Optimizer - A Qiskit Function by Kipu Quantum
description: Efficiently solve optimization problems with Kipu Quantum's Iskay Quantum Optimizer, available on the IBM Qiskit Functions Catalog
---

{/* cspell:ignore Kipu, DCQO, QUBO, HUBO, counterdiabatic, Iskay, bitflips, Cadavid, Chandarana */}

# Iskay Quantum Optimizer - A Qiskit Function by Kipu Quantum

*See the [API reference](/docs/api/functions/kipu-optimization)*

<Admonition type="note" title="Note">
* Qiskit Functions are an experimental feature available only to IBM Quantum&reg; Premium Plan, Flex Plan, and On-Prem (via IBM Quantum Platform API) Plan users. They are in preview release status and subject to change.
</Admonition>

## Overview

With the Iskay Quantum Optimizer by Kipu Quantum, you can tackle complex optimization problems using IBM&reg; quantum computers. This solver leverages Kipu's cutting-edge [bf-DCQO algorithm](https://doi.org/10.48550/arXiv.2409.04477) requiring only the objective function as input to automatically deliver problem solutions. It can handle optimization problems involving up to 156 qubits, enabling the use of all qubits of the IBM quantum devices. The Optimizer uses a 1-to-1 mapping between classical variables and qubits, which allows you to tackle optimization problems with up to 156 binary variables.

The Optimizer enables the solving of unconstrained binary optimization problems. In addition to the commonly used QUBO (Quadratic Unconstrained Binary Optimization) formulation, it also supports higher-order (HUBO) optimization problems. The solver utilizes a non-variational quantum algorithm, performing most of the computation on quantum devices.

The following provides more details about the used algorithm and a brief guide on how to use the function, as well as benchmarking results on various problem instances of different sizes and complexities.

## Description

The Optimizer is a ready-to-use implementation of cutting-edge quantum optimization algorithms. It solves optimization problems by running highly-compressed quantum circuits on quantum hardware. This compression is achieved by introducing counterdiabatic terms into the underlying time evolution of the quantum system. The algorithm executes several iterations of hardware runs to obtain the final solutions and combines it with post-processing. These steps are seamlessly integrated into the Optimizer's workflow and are executed automatically.

### How does the Quantum Optimizer work?
This section outlines the basics of the implemented bf-DCQO algorithm. An introduction to the algorithm can also be found on the [Qiskit YouTube channel](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s).

The algorithm is based on the time evolution of a quantum system which is transformed over time, where the problem solution is encoded in the ground state of the quantum system at the end of the evolution. According to the [adiabatic theorem](https://en.wikipedia.org/wiki/Adiabatic_theorem), this evolution has to be slow to ensure the system remains in its ground state. Digitizing this evolution is the basis of digitized quantum adiabatic computation (DQA) and the infamous QAOA algorithm. However, the required slow evolution is not feasible for increasing problem sizes since it results in an increasing circuit depth. By using counterdiabatic protocols, you can suppress unwanted excitations occurring during short evolution times while remaining in the ground state. Here, digitizing this shorter evolution time results in quantum circuits with shorter depth and fewer entangling gates.

The circuits of the bf-DCQO algorithms typically use up to ten times fewer entangling gates than DQA, and three to four times fewer entangling gates than standard QAOA implementations. Because of the smaller number of gates, fewer errors occur during the circuit execution on hardware. Hence, the optimizer does not require using techniques like error suppression or error mitigation. Implementing them in future versions can enhance the solution quality even further.

Although the bf-DCQO algorithm uses iterations, it is non-variational. After each iteration of the algorithm, the distribution of states is measured. The obtained distribution is used to calculate a so-called bias-field. The bias-field allows starting the next iteration from an energy state near the previously found solution. In this way, the algorithm moves with each iteration to solutions of lower energy. Typically, approximately ten iterations are sufficient to converge to a solution, in total requiring a much lower number of iterations than variational algorithms, which is on the order of  approximately 100 iterations.

The optimizer combines the bf-DCQO algorithm with classical post-processing. After measuring the distribution of states, a local search is performed. During the local search, the bits of the measured solution are randomly flipped. After the flip, the energy of the new bitstring is evaluated. If the energy is lower, the bitstring is kept as the new solution. The local search only scales linearly with the number of qubits; hence, it is computationally cheap. Since the post-processing corrects local bitflips, it compensates for bit-flip errors that often are the result of hardware imperfections and readout errors.

### Workflow

A schematic of the workflow of the Quantum Optimizer follows.

![Workflow](/docs/images/guides/kipu-optimization/workflow.svg "Workflow of the Quantum Optimizer")

By using the Quantum Optimizer, solving an optimization problem on quantum hardware can be reduced to
* Formulate the objective function of the problem
* Access the Optimizer via Qiskit Functions
* Run the Optimizer and collect the result

## Benchmarks

The benchmark metrics below show that the Optimizer effectively addresses problems involving up to 156 qubits and offer a general overview of the optimizer's accuracy and scalability across different problem types. Note that actual performance metrics may vary depending on specific problem characteristics, such as the number of variables, the density and locality of terms in the objective function, and the polynomial order.

The following table includes the approximation ratio (AR), a metric defined as follows:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
where $C$ is the objective function, $C_{\textrm{min}}$, $C_{\textrm{max}}$ are its minimum and maximum values, and $C^{*}$ is the cost of the best solution found, respectively. Therefore, AR=100\% means that the ground state of the problem has been obtained.


| Example           | Number of qubits | Approximation Ratio | Total time (s) | Runtime usage (s) | Total Number of shots | Number of iterations |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| Unweighted MaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| Unweighted MaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| Unweighted MaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- The MaxCut instances with 28, 30, and 32 qubits were run on ibm_sherbrooke. Instances with 80, 100, and 120 were run  on a Heron r2 processor.
- The HUBO instances were also run  on a Heron r2 processor.

All the benchmark instances are accessible on GitHub (see [Kipu benchmark instances](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). An example to run these instances can be found in [Example 3: Benchmark instances](#example-3-benchmark-instances).

## Use cases

Typical use cases for the Optimization solver are combinatorial optimization problems. You can solve problems from many industries like finance, pharmaceutics, or logistics. Some examples follow.
* Portfolio optimization (QUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.22.054037) and [white paper](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Protein folding (HUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Logistics scheduling (QUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Network optimization: [webinar](https://www.youtube.com/watch?v=w5SrCIK88No)
* Market split (QUBO): [tutorial](/docs/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

If you are interested in tackling a specific use case and develop a dedicated mapping, we can assist you. [Contact us](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5).

## Get support

For support, contact support@kipu-quantum.com.

## Next steps

- [Request access to the Quantum Optimizer by Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5).
- Visit the [API reference](/docs/api/functions/kipu-optimization) for this Qiskit Function.
- Try the [Solve the Market Split problem with Kipu Quantum's Iskay Quantum Optimizer](/docs/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer) tutorial.
- Review [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477](https://arxiv.org/abs/2409.04477).
- Review [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898](https://arxiv.org/abs/2405.13898).
- Review [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663](https://arxiv.org/abs/2505.08663).

## Additional information

Iskay, like our company name Kipu Quantum, is a Peruvian word. Although we are a startup from Germany, these words come from the native country of one of our co-founders, where the Quipu was one of the very first calculation machines developed by mankind 2000 years BCE.